# ☁️ Day 151 — Cloud Deployment: Streamlit Community Cloud + Render.com
## Month 8 | Streamlit + FastAPI Deployment | Deepanshu Garg

---

| Field | Details |
|---|---|
| **Month** | 8 — Streamlit + FastAPI Deployment |
| **Day** | 151 (Month 8, Week 4, Day 1) |
| **Topic** | Cloud Deployment · Streamlit Community Cloud · Render.com · Secrets Management · Production `requirements.txt` |
| **Dataset** | FreelanceHub India (500 rows, seed=141) |
| **Deliverables** | `render.yaml` · `runtime.txt` · `.streamlit/secrets.toml` · `.streamlit/config.toml` · Updated `main.py` (health endpoint) · Updated `streamlit_app.py` (st.secrets) · Pinned `requirements.txt` (×2) |
| **Max Score** | 80 pts + 10★ Bonus |
| **GitHub Repo** | Month8-Streamlit-FastAPI-Portfolio |

---

## 📊 Month 8 Scorecard

| Day | Topic | Score |
|---|---|---|
| 141 | Streamlit Fundamentals | 80/80 + 10★ ✅ |
| 142 | Session State + Tabs + Forms | 80/80 + 10★ ✅ |
| 143 | File Upload + Dynamic EDA | 80/80 + 10★ ✅ |
| 144 | ML Model Deployment (Basic) | 80/80 + 10★ ✅ |
| 145 | Advanced Caching + Multi-Page Apps | 80/80 + 10★ ✅ |
| 146 | ML Explainability Dashboard (SHAP) | 80/80 + 10★ ✅ |
| 147 | FastAPI Fundamentals + ML Serving | 80/80 + 10★ ✅ |
| 148 | FastAPI + DB + Auth | 80/80 + 10★ ✅ |
| 149 | Streamlit ↔ FastAPI Full-Stack | 80/80 + 10★ ✅ |
| 150 | Docker + Containerization | 80/80 + 10★ ✅ |
| **151** | **Cloud Deployment** | **← Today** |

**Running Total: 800/800 + 100★ 🔥**

---

## 🎯 Why This Day Matters

Day 150 taught you to containerize your app. Today you **ship it to the internet.**

> **Freelance reality:** Every client asks the same question after you demo your Streamlit app:  
> *"Can I access this from a browser without installing anything?"*  
> Today you answer YES — with a live URL you can paste into a proposal.

**Two free platforms you'll deploy to:**

| Platform | Best For | Free Tier | Docker? |
|---|---|---|---|
| **Streamlit Community Cloud** | Streamlit-only apps | ✅ Unlimited public apps | ❌ Git-based |
| **Render.com** | FastAPI / Docker services | ✅ 750 hrs/month | ✅ Docker-native |

**The deployment stack after today:**
```
GitHub Repo
├── day149_api/          → Deploy to Render.com (Docker)
│   ├── Dockerfile
│   ├── requirements.txt (pinned)
│   └── render.yaml
└── day149_app/          → Deploy to Streamlit Community Cloud (Git)
    ├── streamlit_app.py
    ├── requirements.txt (pinned)
    ├── runtime.txt
    └── .streamlit/
        ├── config.toml
        └── secrets.toml   ← NEVER committed to GitHub
```


---
## 📂 Section 1 — Raw Data
> ⚠️ **NEVER MODIFY THIS SECTION.** Reference only — carried from Day 149/150.


In [ ]:
# Section 1: FreelanceHub India Dataset (seed=141, carried from Day 141)
# Goal: Reconstruct the same dataset used across all Month 8 days
# Method: Fixed seed=141 ensures identical data every run

import pandas as pd
import numpy as np

np.random.seed(141)
n = 500

platforms   = np.random.choice(["Upwork","Fiverr","Toptal","Freelancer","Guru"], n,
                                 p=[0.3,0.25,0.15,0.2,0.1])
categories  = np.random.choice(["ML/AI","Web Dev","Data Analysis","Design","Writing"], n,
                                 p=[0.25,0.3,0.2,0.15,0.1])
experience  = np.random.choice(["Junior","Mid","Senior"], n, p=[0.35,0.40,0.25])
revenue     = np.round(np.random.normal(75000,20000,n).clip(20000,150000), 0)
rating      = np.round(np.random.uniform(3.5,5.0,n), 1)
projects    = np.random.randint(1, 50, n)
on_time_pct = np.round(np.random.uniform(70,100,n), 1)

df_raw = pd.DataFrame({
    "platform":      platforms,
    "category":      categories,
    "experience":    experience,
    "revenue_inr":   revenue,
    "rating":        rating,
    "projects_done": projects,
    "on_time_pct":   on_time_pct
})

print(f"Dataset shape: {df_raw.shape}")
print(df_raw.head())


---
## 📚 Section 2 — Concept Notes

### 2.1 Why Pinned `requirements.txt` Matters in Production

Local dev lets you do `pip install fastapi` — it grabs the latest.  
On a cloud server, "latest" in January ≠ "latest" in June.  
**Unpinned = reproducibility broken = deployment roulette.**

```
# ❌ Development (OK locally, dangerous on server)
fastapi
uvicorn
pandas

# ✅ Production (locked, reproducible)
fastapi==0.111.0
uvicorn[standard]==0.29.0
pandas==2.2.2
```

**How to generate:** `pip freeze > requirements.txt` inside your venv.

---

### 2.2 Streamlit Community Cloud — How It Works

```
GitHub push → Streamlit Cloud detects change → auto-rebuilds app → live URL updates
```

**Key files Streamlit Cloud reads:**
| File | Purpose |
|---|---|
| `requirements.txt` | Python packages to install |
| `runtime.txt` | Python version (e.g., `python-3.11`) |
| `.streamlit/config.toml` | Theme, server settings |
| `.streamlit/secrets.toml` | API keys, URLs — **never commit this** |

**`.gitignore` must contain:**
```
.streamlit/secrets.toml
*.db
*.joblib
__pycache__/
.env
```

---

### 2.3 Render.com — Docker Deployment

Render.com can deploy any Docker container from a GitHub repo.

**Two ways to deploy on Render:**
1. **Web Service** — single Docker container (FastAPI)
2. **render.yaml** — Infrastructure-as-Code, defines all services in one file

```yaml
# render.yaml (root of repo)
services:
  - type: web
    name: freelancehub-api
    env: docker
    dockerfilePath: ./day149_api/Dockerfile
    healthCheckPath: /health
    envVars:
      - key: SECRET_KEY
        value: freelancehub-secret-key-2024
```

**render.yaml key fields:**
| Field | What It Does |
|---|---|
| `type: web` | HTTP-accessible service |
| `env: docker` | Use Dockerfile (not buildpacks) |
| `dockerfilePath` | Relative path to Dockerfile |
| `healthCheckPath` | Render pings this endpoint to verify app is alive |
| `envVars` | Environment variables injected at runtime |

---

### 2.4 `st.secrets` — The Right Way to Handle Credentials

**Never hardcode secrets in Python files — they end up on GitHub.**

```python
# ❌ WRONG — API key visible on GitHub
API_KEY = "freelancehub-secret-key-2024"
API_URL = "http://localhost:8000"

# ✅ RIGHT — secrets stored outside the codebase
API_KEY = st.secrets["api"]["key"]
API_URL = st.secrets["api"]["base_url"]
```

**Local `.streamlit/secrets.toml`:**
```toml
[api]
base_url = "https://freelancehub-api.onrender.com"
key = "freelancehub-secret-key-2024"
```

**On Streamlit Cloud:** Paste the same key-value pairs in the UI (Settings → Secrets).  
The file on disk is never committed.

---

### 2.5 Health Check Endpoint — Why Cloud Platforms Require It

Render, Railway, Fly.io all ping a `/health` endpoint to decide if your container is alive.  
If it returns non-200 → platform restarts the container.

```python
# FastAPI health endpoint
@app.get("/health")
def health_check():
    return {
        "status": "healthy",
        "service": "FreelanceHub API",
        "version": "1.0.0"
    }
```

**Best practice:** Also check DB connectivity inside `/health`.

---

### 2.6 `runtime.txt` for Streamlit Community Cloud

One-line file. Tells Streamlit Cloud which Python version to use.

```
python-3.11
```

Without it → Streamlit Cloud picks its default (may differ from your local version).

---

### 2.7 Deployment Checklist (Every Project)

```
Pre-deployment:
□ requirements.txt has pinned versions
□ runtime.txt specifies Python version
□ .gitignore covers secrets.toml, .env, *.db, *.joblib
□ Hardcoded localhost URLs replaced with st.secrets / env vars
□ /health endpoint returns {"status": "healthy"}
□ Dockerfile uses 0.0.0.0 host (not 127.0.0.1)
□ Docker build tested locally before pushing

Post-deployment:
□ Render /health URL returns 200
□ Streamlit Cloud app loads without import errors
□ Streamlit app correctly calls Render API (not localhost)
□ Live URL tested in incognito (no cached state)
```


---
## 🏋️ Section 3 — Practice Tasks

**You create 7 files today. All are deployment config/code files — no standalone .py to run.**  
Build each file, then paste its contents into the cells below for scoring.

| Task | File | Points |
|---|---|---|
| T1 | Pinned `requirements.txt` for FastAPI service | 10 pts |
| T2 | Pinned `requirements.txt` for Streamlit app | 10 pts |
| T3 | `render.yaml` for Render.com deployment | 15 pts |
| T4 | `runtime.txt` + `.streamlit/config.toml` | 10 pts |
| T5 | `/health` endpoint added to `main.py` | 15 pts |
| T6 | `streamlit_app.py` updated to use `st.secrets` | 15 pts |
| T7 | NRA deployment analysis (written) | 5 pts |
| **Bonus** | `.github/workflows/deploy.yml` — CI health check | 10★ |
| **Total** | | **80 pts + 10★** |

---

### Task 1 — FastAPI Pinned `requirements.txt` (10 pts)
Write `day149_api/requirements.txt` with pinned versions.  
**Must include:** fastapi, uvicorn[standard], pandas, scikit-learn, joblib, pydantic, sqlalchemy, python-jose[cryptography], passlib[bcrypt].


In [ ]:
# Task 1: FastAPI requirements.txt
# Goal: Write pinned requirements.txt for the FastAPI service
# Method: All packages needed by day149_api/main.py, with ==version pinned

# Paste your file contents as a Python string:
fastapi_requirements_txt = """
# WRITE YOUR requirements.txt CONTENT HERE
"""

print(fastapi_requirements_txt)


### Task 2 — Streamlit Pinned `requirements.txt` (10 pts)
Write `day149_app/requirements.txt` with pinned versions.  
**Must include:** streamlit, requests, pandas, plotly, shap, scikit-learn, joblib.


In [ ]:
# Task 2: Streamlit requirements.txt
# Goal: Write pinned requirements.txt for the Streamlit app
# Method: Only packages actually imported by streamlit_app.py

streamlit_requirements_txt = """
# WRITE YOUR requirements.txt CONTENT HERE
"""

print(streamlit_requirements_txt)


### Task 3 — `render.yaml` (15 pts)
Write a `render.yaml` at the repo root that defines:
- A `web` service named `freelancehub-api`
- Environment: docker
- Dockerfile path: `./day149_api/Dockerfile`
- Health check path: `/health`
- `SECRET_KEY` env var
- A `plan: free` field


In [ ]:
# Task 3: render.yaml
# Goal: Infrastructure-as-Code file for Render.com deployment
# Method: YAML format, defines Docker web service with health check

render_yaml = """
# WRITE YOUR render.yaml CONTENT HERE
"""

print(render_yaml)


### Task 4 — `runtime.txt` + `.streamlit/config.toml` (10 pts)
**Part A:** Write `runtime.txt` specifying Python 3.11.  
**Part B:** Write `.streamlit/config.toml` with:
- `[theme]` section: primaryColor `#1F3864`, backgroundColor `#FFFFFF`, font `"sans serif"`
- `[server]` section: `headless = true`, `port = 8501`


In [ ]:
# Task 4A: runtime.txt
runtime_txt = """
# WRITE YOUR runtime.txt CONTENT HERE
"""

# Task 4B: .streamlit/config.toml
config_toml = """
# WRITE YOUR config.toml CONTENT HERE
"""

print("=== runtime.txt ===")
print(runtime_txt)
print("=== .streamlit/config.toml ===")
print(config_toml)


### Task 5 — `/health` Endpoint in `main.py` (15 pts)
Add a `/health` GET endpoint to `day149_api/main.py` that:
1. Returns `{"status": "healthy", "service": "FreelanceHub API", "version": "1.0.0"}`
2. Also checks DB connectivity — tries a simple `SELECT 1` on the SQLite DB
3. Returns `{"status": "degraded", "db": "unreachable"}` if DB check fails
4. Returns HTTP 503 on failure (use `HTTPException` or `Response` with `status_code=503`)

Paste the **entire health endpoint function** (not the full file).


In [ ]:
# Task 5: /health endpoint
# Goal: Cloud-platform-compatible health check endpoint
# Method: FastAPI GET route that checks service + DB status

# Paste your health endpoint code here:
health_endpoint_code = '''
# WRITE YOUR /health ENDPOINT HERE
'''

print(health_endpoint_code)


### Task 6 — `streamlit_app.py` Updated for `st.secrets` (15 pts)
Update the relevant sections of `streamlit_app.py` to:
1. Replace hardcoded `API_BASE_URL = "http://localhost:8000"` with `st.secrets["api"]["base_url"]`
2. Replace hardcoded `API_KEY = "..."` with `st.secrets["api"]["key"]`
3. Add a `try/except` block around the secrets access with a fallback to env vars:
   ```python
   try:
       API_BASE_URL = st.secrets["api"]["base_url"]
   except Exception:
       API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
   ```
4. Write the `.streamlit/secrets.toml` structure (the local version, never committed)

Paste the updated config section + secrets.toml structure.


In [ ]:
# Task 6: st.secrets integration
# Goal: Remove all hardcoded credentials from streamlit_app.py
# Method: st.secrets with os.getenv fallback for local dev

# Part A: Updated config section in streamlit_app.py
secrets_integration_code = '''
# WRITE YOUR UPDATED CONFIG SECTION HERE
'''

# Part B: .streamlit/secrets.toml structure
secrets_toml_content = 

print("=== streamlit_app.py config section ===")
print(secrets_integration_code)
print("=== .streamlit/secrets.toml ===")
print(secrets_toml_content)


### Task 7 — NRA Deployment Analysis (5 pts)
Using the FreelanceHub dataset, compute platform-level revenue.  
Write **one NRA insight** explaining which platform to deploy for (based on revenue concentration).

**Format:** Number + Reason + Action


In [ ]:
# Task 7: NRA Deployment Analysis
# Goal: Use data to justify deployment investment
# Method: Platform avg revenue → NRA format
# IMPORTANT: Run ALL columns in exact order to match Section 1 dataset

import pandas as pd
import numpy as np

np.random.seed(141)
n = 500
platforms   = np.random.choice(["Upwork","Fiverr","Toptal","Freelancer","Guru"], n, p=[0.3,0.25,0.15,0.2,0.1])
categories  = np.random.choice(["ML/AI","Web Dev","Data Analysis","Design","Writing"], n, p=[0.25,0.3,0.2,0.15,0.1])
experience  = np.random.choice(["Junior","Mid","Senior"], n, p=[0.35,0.40,0.25])
revenue     = np.round(np.random.normal(75000,20000,n).clip(20000,150000), 0)
rating      = np.round(np.random.uniform(3.5,5.0,n), 1)
projects    = np.random.randint(1, 50, n)
on_time_pct = np.round(np.random.uniform(70,100,n), 1)

df = pd.DataFrame({
    "platform":      platforms,
    "category":      categories,
    "experience":    experience,
    "revenue_inr":   revenue,
    "rating":        rating,
    "projects_done": projects,
    "on_time_pct":   on_time_pct
})

# Compute platform stats — run this cell, READ the output, THEN write your NRA
platform_stats = df.groupby("platform")["revenue_inr"].agg(["mean","count"]).round(0)
platform_stats.columns = ["avg_revenue", "freelancer_count"]
platform_stats = platform_stats.sort_values("avg_revenue", ascending=False)

print(platform_stats)
print(f"\nOverall avg: ₹{df['revenue_inr'].mean():,.0f}")

# WRITE YOUR NRA INSIGHT BELOW (use numbers from the printed output above):
nra_insight = """
Number  : 
Reason  : 
Action  : 
"""
print(nra_insight)


### ⭐ Bonus Task — GitHub Actions CI Health Check (10★)
Write a `.github/workflows/deploy-check.yml` that:
1. Triggers on `push` to `main`
2. Builds the FastAPI Docker image
3. Runs `docker run -d -p 8000:8000 freelancehub-api`
4. Waits 5 seconds then `curl -f http://localhost:8000/health`
5. Fails the workflow if health check returns non-200

Paste the full YAML.


In [ ]:
# Bonus: .github/workflows/deploy-check.yml
# Goal: Automated CI that verifies health endpoint after Docker build
# Method: GitHub Actions workflow with docker build + curl health check

github_actions_yaml = """
# WRITE YOUR GitHub Actions YAML HERE
"""

print(github_actions_yaml)


---
## 📋 Section 4 — Scoring Rubric

### Task Breakdown

| Task | What Is Checked | Points |
|---|---|---|
| **T1** FastAPI `requirements.txt` | All 9 packages present · All pinned with `==` · No extras that break builds | 10 |
| **T2** Streamlit `requirements.txt` | All 7 packages present · All pinned · No test/dev packages | 10 |
| **T3** `render.yaml` | `type: web` ✓ · `env: docker` ✓ · `dockerfilePath` correct ✓ · `healthCheckPath: /health` ✓ · `envVars` includes SECRET_KEY ✓ | 15 |
| **T4** `runtime.txt` + `config.toml` | `python-3.11` exact format (2 pts) · `[theme]` with primaryColor `#1F3864` (4 pts) · `[server]` with `headless=true` + `port=8501` (4 pts) | 10 |
| **T5** `/health` endpoint | Route is GET `/health` ✓ · Returns correct JSON keys ✓ · DB check via `SELECT 1` ✓ · Returns 503 + `"status":"degraded"` on failure ✓ | 15 |
| **T6** `st.secrets` | Replaces both hardcoded values ✓ · `try/except` with `os.getenv` fallback ✓ · `secrets.toml` has correct `[api]` section ✓ · `secrets.toml` noted as never-committed ✓ | 15 |
| **T7** NRA Analysis | Number from printed output (not memory) ✓ · Reason explains business logic ✓ · Action is specific and committal ✓ | 5 |
| **⭐ Bonus** GitHub Actions | All 5 steps present · `curl -f` health check correct · Cleanup step present | 10★ |

---

### Deduction Rules

| Error Type | Deduction |
|---|---|
| Missing package in requirements.txt | -2 pts per package |
| Any package unpinned (no `==version`) | -1 pt per package |
| `render.yaml` missing `healthCheckPath` | -5 pts |
| `render.yaml` env is `python` not `docker` | -5 pts |
| Health endpoint missing DB check | -5 pts |
| Health endpoint doesn't return 503 on failure | -4 pts |
| `st.secrets` has no `try/except` fallback | -5 pts |
| `secrets.toml` shows hardcoded values committed to repo | -5 pts (security flag) |
| NRA number not from printed output | -3 pts |
| NRA action is hedging ("consider", "may") | -1 pt |

---

### ⭐ Bonus Deductions

| Error | Deduction |
|---|---|
| Missing `curl -f` (no failure on non-200) | -4★ |
| No cleanup step (`docker rm`) | -2★ |
| Triggers only on push, not PR | -2★ |
| Wrong Docker build path | -2★ |

---

### Interview Framing

**Q: "You say your app is deployed — how do I know it won't break when I open it tomorrow?"**

**A:** "The app has a `/health` endpoint that the cloud platform pings every 60 seconds. If it returns non-200, the platform automatically restarts the container. On top of that, every push to GitHub triggers a CI workflow that builds a fresh Docker image and runs the health check — so broken code never reaches production. The Streamlit frontend uses `st.secrets` so no credentials are ever in the codebase."

---

### Key Takeaway

> **Deployment without a health check is a liability, not a product.**  
> A live URL impresses clients for 5 minutes. A live URL that *stays up*, *handles secrets properly*, and *fails gracefully with a 503* earns trust for 5 months.  
> Every professional deployment has three things: pinned dependencies, secrets management, and a health endpoint. Today you built all three.
